In [2]:
from langgraph.graph import StateGraph,START, END , MessagesState
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI 
from dotenv import load_dotenv
from typing import TypedDict
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain_core.messages.utils import trim_messages 
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

c:\Users\dell\Desktop\Langgraph-learning\agent\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.7) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\dell\Desktop\Langgraph-learning\agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

In [4]:
def token_counter(messages):
    total_chars = 0
    
    for msg in messages:
        total_chars += len(msg.content)
    
    # Approx: 1 token ≈ 4 chars
    return total_chars // 4

In [5]:
class ChatState(MessagesState):
    summary: str

In [6]:
def summarize_conversation(state: ChatState):

    existing_summary = state["summary"]

    # Build summarization prompt
    if existing_summary:
        prompt = (
            f"Existing summary:\n{existing_summary}\n\n"
            "Extend the summary using the new conversation above."
        )
    else:
        prompt = "Summarize the conversation above."

    messages_for_summary = state["messages"] + [
        HumanMessage(content=prompt)
    ]

    response = llm.invoke(messages_for_summary)

    # Keep only last 2 messages verbatim
    messages_to_delete = state["messages"][:-2]

    return {
        "summary": response.content,
        "messages": [RemoveMessage(id=m.id) for m in messages_to_delete],
    }

In [7]:
def chat_node(state: ChatState):
    messages = []

    if state["summary"]:
        messages.append({
            "role": "system",
            "content": f"Conversation summary:\n{state['summary']}"
        })

    messages.extend(state["messages"])

    print(messages)

    response = llm.invoke(messages)
    return {"messages": [response]}

In [8]:
def should_summarize(state: ChatState):
    return len(state["messages"]) > 6

In [9]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)
graph.add_node("chat",chat_node)
graph.add_node("summarize", summarize_conversation)
graph.add_edge(START,"chat")
graph.add_conditional_edges("chat", should_summarize,{ True :"summarize", False :END})
graph.add_edge("summarize",END)
workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config = {"configurable": {"thread_id": "t1"}}

def run_turn(text: str):
    out = workflow.invoke({"messages": [HumanMessage(content=text)], "summary": ""}, config=config)
    return out

In [18]:
def show_state():
    snap = workflow.get_state(config)
    vals = snap.values
    print("\n--- STATE ---")
    print("summary:", vals.get("summary", ""))
    print("num_messages:", len(vals.get("messages", [])))
    print("messages:")
    for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content[:80])

In [19]:
run_turn('Quantum Physics')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='25e533af-170f-4321-8d03-478925722eec'), AIMessage(content='Quantum Physics is a branch of science that deals with the behavior of matter and energy at the molecular, atomic, nuclear, and even smaller levels. It\'s a fundamental theory in physics that describes nature at the smallest scales of energy levels of atoms and subatomic particles.\n\nIt\'s famously known for being counter-intuitive and challenging our everyday understanding of reality, as the rules that govern the macroscopic world (classical physics) break down at the quantum level.\n\nHere are some of the core concepts and characteristics of Quantum Physics:\n\n1.  **Quantization:**\n    *   One of the most fundamental ideas. It means that certain physical properties (like energy, momentum, angular momentum, etc.) don\'t exist on a continuous spectrum but rather in discrete, indivisible packets called "quanta."\n    *   For example, ligh

In [20]:
run_turn('how is albert einstein related to quantum physics?')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='25e533af-170f-4321-8d03-478925722eec'), AIMessage(content='Quantum Physics is a branch of science that deals with the behavior of matter and energy at the molecular, atomic, nuclear, and even smaller levels. It\'s a fundamental theory in physics that describes nature at the smallest scales of energy levels of atoms and subatomic particles.\n\nIt\'s famously known for being counter-intuitive and challenging our everyday understanding of reality, as the rules that govern the macroscopic world (classical physics) break down at the quantum level.\n\nHere are some of the core concepts and characteristics of Quantum Physics:\n\n1.  **Quantization:**\n    *   One of the most fundamental ideas. It means that certain physical properties (like energy, momentum, angular momentum, etc.) don\'t exist on a continuous spectrum but rather in discrete, indivisible packets called "quanta."\n    *   For example, ligh

In [21]:
run_turn('explain theory of relativity in 2 sentences')
show_state()

[HumanMessage(content='Quantum Physics', additional_kwargs={}, response_metadata={}, id='25e533af-170f-4321-8d03-478925722eec'), AIMessage(content='Quantum Physics is a branch of science that deals with the behavior of matter and energy at the molecular, atomic, nuclear, and even smaller levels. It\'s a fundamental theory in physics that describes nature at the smallest scales of energy levels of atoms and subatomic particles.\n\nIt\'s famously known for being counter-intuitive and challenging our everyday understanding of reality, as the rules that govern the macroscopic world (classical physics) break down at the quantum level.\n\nHere are some of the core concepts and characteristics of Quantum Physics:\n\n1.  **Quantization:**\n    *   One of the most fundamental ideas. It means that certain physical properties (like energy, momentum, angular momentum, etc.) don\'t exist on a continuous spectrum but rather in discrete, indivisible packets called "quanta."\n    *   For example, ligh